## 持久化

In [ ]:
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

#### 异常恢复

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from typing import TypedDict
import time

class State(TypedDict):
    topic: str
    poem: str
    joke: str
    output: str

class InputState(TypedDict):
    topic: str

class OutputState(TypedDict):
    output: str

def node_poem(state: InputState):
    poem_msg = HumanMessage(f"请写一首关于{state["topic"]}的五言绝句，不用解释内容")
    # time.sleep(2)
    # raise Exception("模拟异常")
    ai_msg = model.invoke([poem_msg])
    return {
        "poem": ai_msg.content,
    }

def node_joke(state: InputState):
    joke_msg = HumanMessage(f"请写一个关于{state["topic"]}的超短笑话")
    ai_msg = model.invoke([joke_msg])
    return {
        "joke": ai_msg.content,
    }

def node_output(state: State):
    return {
        "output": f"主题：{state["topic"]}\n\n五言绝句：{state["poem"]}\n\n笑话：{state["joke"]}"
    }


builder = StateGraph(State)
builder.add_node(node_poem)
builder.add_node(node_joke)
builder.add_node(node_output)
builder.add_edge(START, "node_poem")
builder.add_edge(START, "node_joke")
builder.add_edge("node_poem", "node_output")
builder.add_edge("node_joke", "node_output")
builder.add_edge("node_output", END)
graph = builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "4-3"}}
result = graph.invoke({"topic": "猫"}, config=config)
print(result["output"])

#### 打印状态历史

从打印的状态历史中可以看到，调node_poem节点时，抛出了异常

In [ ]:
from rich.pretty import pprint

history_checkpoint = list(graph.get_state_history(config=config))
pprint(history_checkpoint)

#### 恢复运行

注释异常的代码，调用参数为`None`，再次调用，可以看到，从之前错误的状态开始运行

In [ ]:
config = {"configurable": {"thread_id": "4-3"}}
result = graph.invoke(None, config=config)
print(result["output"])

In [ ]:
from rich.pretty import pprint

history_checkpoint = list(graph.get_state_history(config=config))
pprint(history_checkpoint)